# `integrate_results.csv`

Connor Shrader / 09-21-2025

This file loads in and computes metrics on all of the clone results and saves to the file `all_clone_results.csv`.

In [1]:
import os
import pandas as pd
import numpy as np
import ast
import testis_modeling as tm
from tqdm import tqdm

from saving_loading import find_scenarios, load_all_scenarios, load_clone_names
from metrics import init_time, last_time, max_time, freq_at, size, avg_freq_change
from metrics import run_likelihood

In [2]:
scenarios = find_scenarios()
fish_data, clone_data = load_all_scenarios(scenarios)
clone_names = load_clone_names()

In [3]:
print(f'Found scenarios are: {scenarios}.')

Found scenarios are: ['n500_r15_y6', 'n500_r1_y6', 'n1500_r36_y6', 'n1000_r1_y4', 'n500_r15_y9', 'n500_r1_y9', 'n1500_r36_y9', 'n1000_r0_y4', 'n1000_r15_y6', 'n500_r4_y6', 'n500_r0_y9', 'n1000_r4_y4', 'n1000_r15_y9', 'n500_r0_y6', 'n500_r4_y9', 'n1000_r36_y4', 'n1500_r4_y9', 'n1500_r0_y6', 'real', 'n1500_r0_y9', 'n1500_r4_y6', 'n1000_r3_y6', 'n500_r36_y4', 'n1500_r1_y9', 'n1000_r2_y6', 'n1500_r15_y4', 'n1500_r1_y6', 'n1000_r1_y6', 'n1500_r36_y4', 'n500_r1_y4', 'n500_r15_y4', 'n1000_r1_y9', 'n500_r0_y4', 'n1000_r4_y6', 'n1000_r0_y9', 'n500_r4_y4', 'n1000_r15_y4', 'n1000_r0_y6', 'n1000_r4_y9', 'n1500_r4_y4', 'n1000_r36_y9', 'n1500_r0_y4', 'n1000_r36_y6', 'n1500_r15_y9', 'n1000_r2.5_y6', 'n1500_r1_y4', 'n500_r36_y9', 'n1500_r15_y6', 'n500_r36_y6'].


In [4]:
long_dfs = []
for scenario in scenarios:
    path = os.path.join('scenarios', scenario, f'{scenario}_long.csv')
    if os.path.isfile(path):
        long_dfs.append(pd.read_csv(path))

In [5]:
long_results = pd.concat(long_dfs).set_index(['scenario', 'fish', 'clone'])
print(f'Long results has length {len(long_results)}.')
long_results.head()

Long results has length 39030.


init_time  init_freq        r_opt    if_opt  \
scenario    fish clone                                                
n500_r15_y6 0    0              1   0.304582    21.919817  0.307162   
                 1              1   0.156334    24.455399  0.152912   
                 2              1   0.115903    47.988718  0.117011   
                 3              1   0.094340  1000.000000  0.094286   
                 4              1   0.088949    50.575174  0.088244   

                            l_max      lower       upper   if_null      l_null  
scenario    fish clone                                                          
n500_r15_y6 0    0     -88.556002   8.503530   68.702417  0.289608 -251.078272  
                 1     -20.549349   5.447543  146.436829  0.023360 -249.564210  
                 2     -12.777606   7.574745  473.370544  0.015274 -208.366860  
                 3      -2.947001  20.938202         NaN  0.005241 -105.426060  
                 4      -7.588773   6.353441         NaN  0.006439 -107.163524

In [6]:
# Specify order of columns
columns = ['scenario', 'fish', 'clone', 'init_time', 'init_freq', 'last_time',
           'max_time', 'max_freq', 'size', 'avg_freq_change', 'r_opt', 'if_opt',
           'l_max', 'lower', 'upper', 'if_null', 'l_null']

rows = []
for (scenario, fish, clone), data in tqdm(clone_data.items()):
    if (scenario, fish, clone) in long_results.index:
        row = long_results.loc[scenario, fish, clone].to_dict()
    else:
        row = {col: pd.NA for col in columns}
        it = init_time(data)
        if it is None: continue
        else: row['init_time'] = it
        row['init_freq'] = freq_at(data, row['init_time'])

    row['scenario'] = scenario
    row['fish'] = fish
    row['clone'] = clone
    
    row['last_time'] = last_time(data)
    row['max_time'] = max_time(data)
    row['max_freq'] = freq_at(data, row['max_time'])
    row['size'] = size(data)
    row['avg_freq_change'] = avg_freq_change(data)
    
    rows.append(row)

100%|██████████████████████████████████████████| 48306/48306 [01:24<00:00, 572.46it/s]


In [7]:
df = pd.DataFrame(rows, columns = columns)
df.to_csv('all_clone_results.csv', index = False)

In [8]:
df.query('scenario == "n1000_r3_y6"')

,scenario,fish,clone,init_time,init_freq,last_time,max_time,max_freq,size,avg_freq_change,r_opt,if_opt,l_max,lower,upper,if_null,l_null
19089,n1000_r3_y6,0,0,5.0,0.195867,26.0,26.0,0.228212,0.182240,-0.001540,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
19090,n1000_r3_y6,0,1,5.0,0.185984,26.0,26.0,0.184187,0.220425,0.000086,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
19091,n1000_r3_y6,0,2,5.0,0.138365,26.0,26.0,0.279425,0.123790,-0.006717,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
19092,n1000_r3_y6,0,3,5.0,0.128482,26.0,26.0,0.037736,0.148298,0.004321,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
19093,n1000_r3_y6,0,4,5.0,0.119497,26.0,26.0,0.048518,0.110911,0.003380,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19978,n1000_r3_y6,99,4,5.0,0.090746,26.0,26.0,0.011680,0.065638,0.003765,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
19979,n1000_r3_y6,99,5,5.0,0.054807,26.0,26.0,0.079964,0.048967,-0.001198,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
19980,n1000_r3_y6,99,6,5.0,0.054807,26.0,26.0,0.042228,0.078616,0.000599,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
19981,n1000_r3_y6,99,7,5.0,0.066487,14.0,26.0,0.000000,0.013377,0.006689,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
